# Esteira de Aprendizado de Máquina — Previsão de Risco de Crédito

Este projeto implementa uma esteira completa de aprendizado de máquina para prever se um cliente tem risco de inadimplência grave (atraso de 90 dias ou mais) nos próximos 2 anos.

## 1 e 2. Escolha da Base de Dados e Carregamento

Para este problema de classificação, foi escolhida a base **Give Me Some Credit** disponível no [Kaggle Datasets](https://www.kaggle.com/datasets/lihxlhx/give-me-some-credit/data). O arquivo de treino é baixado e carregado automaticamente na pasta do projeto através da biblioteca `kagglehub`.

In [ ]:
import os
import shutil
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Baixa o dataset via kagglehub
caminho_cache = kagglehub.dataset_download("lihxlhx/give-me-some-credit")

# 2. Localiza o arquivo de treino e copia para a pasta atual do notebook
arquivos = [f for f in os.listdir(caminho_cache) if 'train' in f.lower() and f.endswith('.csv')]
nome_arquivo = arquivos[0]

caminho_origem = os.path.join(caminho_cache, nome_arquivo)
caminho_destino = os.path.join(os.getcwd(), nome_arquivo)

shutil.copy2(caminho_origem, caminho_destino)

# 3. Carrega o DataFrame a partir do arquivo salvo na pasta local
df = pd.read_csv(caminho_destino)

print(f"Arquivo salvo em: {caminho_destino}")
df.head()

## 3. Estatísticas Descritivas Gerais

Nesta etapa foi analisada a estrutura do arquivo considerando o total de linhas e colunas, os tipos de dados, os valores ausentes ou nulos e o desbalanceamento da base, considerando que cerca de 93% dos registros coletados no dataset são de bons pagadores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Informações Gerais e Tipos de Dados ---")
print(df.info())

print("\n--- Contagem de Valores Ausentes (Nulos) por Coluna ---")
print(df.isnull().sum())

print("\n--- Estatísticas Descritivas ---")
display(df.describe().T)

print("\n--- Distribuição da Variável Alvo (SeriousDlqin2yrs) ---")
print("0 = Bom Pagador | 1 = Inadimplente (Atraso >= 90 dias nos próximos 2 anos)")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True) * 100)

## 4. Transformação nas Colunas

Nesta etapa foram feitas três alterações nas colunas: a remoção da coluna de identificação (`Unnamed: 0`) por não ajudar na previsão, o preenchimento dos valores nulos de renda e dependentes usando a mediana para não sofrer distorção com valores muito altos, e a criação da coluna `IncomePerPerson`, que calcula a renda dividida pelo total de pessoas na família.

In [ ]:
# 1. Removendo coluna de identificação sem valor preditivo
cols_id = [col for col in ['Id', 'Unnamed: 0'] if col in df.columns]
df = df.drop(columns=cols_id)

# 2. Preenchendo valores ausentes nas colunas com a mediana
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

# 3. Engenharia de Atributos: Criando coluna de Renda per Capita familiar
df['IncomePerPerson'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

print("Colunas após transformação:", list(df.columns))
print("Total de valores nulos restantes:", df.isnull().sum().sum())

## 5. Transformação nas Linhas

Para limpar os registros inconsistentes da base, foi removido as linhas duplicadas, os cadastros com idade inválida (menor que 18 anos) e os registros que apresentavam os valores `96` e `98` nas colunas de histórico de atraso. Na documentação dessa base, esses números representam códigos internos de erro do sistema — já que seria inviável um cliente acumular 98 atrasos mensais em um período de 2 anos —, portanto foram excluídos para não distorcer o aprendizado do modelo.

In [ ]:
linhas_antes = df.shape[0]

# 1. Remoção de linhas duplicadas
df = df.drop_duplicates().copy()

# 2. Remoção de registros com idade inválida (menor que 18 anos, ex: age == 0)
df = df[df['age'] >= 18].copy()

# 3. Remoção de códigos de erro sistêmico (96 e 98) nas colunas de contagem de atrasos
df = df[df['NumberOfTime30-59DaysPastDueNotWorse'] < 90].copy()
df = df[df['NumberOfTime60-89DaysPastDueNotWorse'] < 90].copy()
df = df[df['NumberOfTimes90DaysLate'] < 90].copy()

linhas_depois = df.shape[0]
print(f"Linhas antes: {linhas_antes} | Linhas depois: {linhas_depois} | Linhas removidas: {linhas_antes - linhas_depois}")

## 6. Divisão em Treinamento, Validação e Teste

Com a base limpa, os dados foram divididos mantendo a mesma proporção de inadimplentes em três partes: 70% para treinamento, 15% para validação e 15% para teste. Também foi aplicado o `RobustScaler` para colocar os dados na mesma escala, ajustado apenas no conjunto de treino para evitar vazamento de informações.

In [ ]:
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

# 1ª Divisão: 70% Treino e 30% Temporário (Validação + Teste)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 2ª Divisão: Divide os 30% restantes ao meio (15% Validação e 15% Teste)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Padronização de escala (fit apenas no Treino para evitar data leakage)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Conjunto de Treinamento : {X_train.shape[0]} amostras")
print(f"Conjunto de Validação   : {X_val.shape[0]} amostras")
print(f"Conjunto de Teste       : {X_test.shape[0]} amostras")

## 7. Treinamento e Avaliação do Modelo

Nesta etapa foi treinado um modelo de Floresta Aleatória (`RandomForestClassifier`) com balanceamento de peso entre as classes para lidar com a diferença entre bons pagadores e inadimplentes, alcançando cerca de 81,38% de acurácia na avaliação preliminar com os dados de validação.

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)
modelo.fit(X_train_scaled, y_train)

y_val_pred = modelo.predict(X_val_scaled)
acc_val = accuracy_score(y_val, y_val_pred)
print(f"Acurácia no Conjunto de Validação: {acc_val:.4f} ({acc_val*100:.2f}%)")

## 8. Matriz de Confusão e Acurácia Final

Aqui o modelo é avaliado no conjunto de teste, atingindo 81,12% de acurácia final. A matriz de confusão gerada permite visualizar os acertos e erros por classe, mostrando que o modelo consegue identificar 73% dos clientes inadimplentes.

In [ ]:
y_test_pred = modelo.predict(X_test_scaled)

acc_test = accuracy_score(y_test, y_test_pred)
print(f"Acurácia Final (Conjunto de Teste): {acc_test:.4f} ({acc_test*100:.2f}%)\n")

print("Relatório de Classificação:\n", classification_report(y_test, y_test_pred))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adimplente (0)', 'Inadimplente (1)'],
            yticklabels=['Adimplente (0)', 'Inadimplente (1)'])
plt.title('Matriz de Confusão - Risco de Crédito (Teste)')
plt.xlabel('Classe Predita pelo Modelo')
plt.ylabel('Classe Real')
plt.show()

## 9. Predição do Modelo Implantado

Por fim, é simulada a chegada de um novo cliente. Os dados passam pela mesma transformação de escala utilizada no treinamento e o modelo retorna a classificação de risco, a probabilidade de inadimplência e a comparação com o status real do cliente.

In [ ]:
novo_cliente = X_test.iloc[[0]]
gabarito_real = y_test.iloc[0]

novo_cliente_scaled = scaler.transform(novo_cliente)

predicao = modelo.predict(novo_cliente_scaled)[0]
probabilidades = modelo.predict_proba(novo_cliente_scaled)[0]

status_predito = "ALTO RISCO - Inadimplente (1)" if predicao == 1 else "BAIXO RISCO - Bom Pagador (0)"
status_real = "Inadimplente (1)" if gabarito_real == 1 else "Bom Pagador (0)"

print("=== SIMULAÇÃO DE PREDIÇÃO EM PRODUÇÃO ===")
display(novo_cliente)
print(f"Decisão do Modelo             : {status_predito}")
print(f"Probabilidade de Inadimplência: {probabilidades[1]*100:.2f}%")
print(f"Status Real do Cliente        : {status_real}")